# Grover Search Simulation

**Day 26 | 30-Day Quantum Computing Challenge**

## Aim:
To implement a two-qubit Grover search and demonstrate amplification of a marked state's probability.

## Tools:
Python, NumPy, Matplotlib (optional Qiskit)

## Expected Result:
One ideal Grover iteration identifies a single marked state among four possibilities with probability 100%.

**Execution:** Run the cells from top to bottom. The notebook uses an ideal simulator unless explicitly stated otherwise.

## 1. How it works
1. Hadamard gates prepare the equal superposition.
2. The **oracle** negates the target amplitude without changing its probability yet.
3. The **diffusion operator** reflects amplitudes about their mean, increasing the target amplitude.
4. Measurement returns the target with probability **1 in this ideal four-state, single-target example after exactly one iteration**.

This is an **ideal statevector simulation**, not a hardware demonstration or a speed benchmark.

## 2. Run the complete experiment
Only `numpy` and `matplotlib` are required. The random seed makes the sample counts reproducible.

In [ ]:
# Experiment step: execute this cell and inspect its output.
import numpy as np
import matplotlib.pyplot as plt

labels = ["00", "01", "10", "11"]
target = "10"
shots = 1000
seed = 26

# Two qubits after Hadamard gates: equal amplitudes for all four states.
uniform = np.ones(4, dtype=complex) / 2
oracle = np.eye(4, dtype=complex)
oracle[labels.index(target), labels.index(target)] = -1

diffusion = 2 * np.outer(uniform, uniform.conj()) - np.eye(4)
state_after_oracle = oracle @ uniform
state_after_grover = diffusion @ state_after_oracle

before_probabilities = np.abs(uniform)**2
after_probabilities = np.abs(state_after_grover)**2

print("Initial amplitudes:", uniform)
print("After oracle:", state_after_oracle)
print("After diffusion:", state_after_grover)
print("Final probabilities:", dict(zip(labels, after_probabilities.round(6))))
assert np.isclose(after_probabilities.sum(), 1.0)
assert np.isclose(after_probabilities[labels.index(target)], 1.0)

rng = np.random.default_rng(seed)
before_counts = rng.multinomial(shots, before_probabilities)
after_counts = rng.multinomial(shots, after_probabilities)
print("Before counts:", dict(zip(labels, before_counts)))
print("After counts:", dict(zip(labels, after_counts)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, counts, title in zip(axes, [before_counts, after_counts],
                             ["Before Grover", "After one Grover iteration"]):
    bars = ax.bar(labels, counts / shots, color=["#7d91ca" if s != target else "#d28c3f" for s in labels])
    ax.set_ylim(0, 1.08)
    ax.set_xlabel("Measured bitstring")
    ax.set_ylabel("Frequency")
    ax.set_title(title)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, count/shots + .025,
                str(count), ha="center")
plt.tight_layout()
plt.savefig("results.png", dpi=180, bbox_inches="tight")
plt.show()


## 3. Why the result is 100% here
The initial amplitudes are `[0.5, 0.5, 0.5, 0.5]`. Marking `10` makes them `[0.5, 0.5, -0.5, 0.5]`. Their mean is `0.25`; diffusion maps each amplitude `a` to `2 × 0.25 − a`, yielding `[0, 0, 1, 0]`.

**Limitations:** Exact success after one iteration is special to four states and one marked item in the noiseless ideal model. Larger searches typically need multiple iterations; real devices are noisy.

## 4. Optional real quantum circuit simulation
The next cell is optional and requires Qiskit Aer. It uses Qiskit bitstring order `q1q0`, matching the target `10`.

In [ ]:
# OPTIONAL: Qiskit implementation (requires qiskit and qiskit-aer).
# Install: %pip install qiskit qiskit-aer
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)
qc.h([0, 1])
# Mark |10> in Qiskit's displayed bitstring convention q1 q0.
# Apply X to q0, then CZ, then undo X on q0.
qc.x(0)
qc.cz(0, 1)
qc.x(0)
# Diffusion: H^2 X^2 CZ X^2 H^2 (up to a global phase).
qc.h([0, 1])
qc.x([0, 1])
qc.cz(0, 1)
qc.x([0, 1])
qc.h([0, 1])
qc.measure([0, 1], [0, 1])
print(qc.draw(output="text"))
print(AerSimulator().run(qc, shots=1000, seed_simulator=26).result().get_counts())


## 5. Experiments to try
- Change `target` to `00`, `01`, or `11` and rerun the NumPy code.
- Increase `shots` and compare sampling fluctuations **before** Grover.
- Remove the oracle or diffusion step and observe what changes.
- Explain why a phase change alone cannot be seen by direct measurement.

## Observation:
Record the measured or simulated outputs and compare them with the theoretical expectation. Note that finite-shot results can fluctuate.

## Conclusion:
The experiment demonstrates the quantum circuit or state transformation described above using simulation; it does not establish performance on physical quantum hardware.